Here is how it works:
- Full Clone and extract all config files: .yml, .yaml and related .json, .sh
- Shallow Clone from all branches: afect number commits & contributors build script or config will be only from the latest snapshop

- it sorts and index the url also save the list of random sample url indeces
- after each puase or interuption the "Cloned Repo" should be emptyied
- by a new new rerun it continues the review from the last reviewed url which is log is stored in .evn by START_NUMBER
- the sample repos are stored in "Cloned_Sample"
- this will save the sample repos as well as metrics, configs, builds and test lines
- Full Clone helps to extract full contributors and commit history
- saving metadata happend immediately so it will get lost by pause/start
Extre feature in v2.0:
- it does compare the downloaded yml files with the list from the previous step


In [ ]:
# -*- coding: utf-8 -*-
"""
Clone-only extractor for Android instrumentation-testing related files.

What this does now (aligned to the finalized logic):
- Auto-detect default branch via: git ls-remote --symref <repo> HEAD
- Shallow clone that branch (--depth 1)
- Extract into TWO buckets only:
    - All_Config_Files:
        * CI YAML (STRICT provider locations only)
        * Shell/runner scripts: .sh .bash .zsh .ksh .bat .cmd .ps1 .psm1 .psd1 + Makefile/makefile/GNUmakefile
        * Gradle & settings: *.gradle *.gradle.kts gradle.properties settings.gradle(.kts)
        * AndroidManifest.xml (ANY path) **only if** it contains at least one <activity
        * Likely CI JSON by allowlist: android-studio-loading.json, saucectl.config.json, firebase.json, test-lab.json
        * Flutter config: pubspec.yaml
    - All_Test_Files:
        * Native Android instrumentation sources: **any** src/**AndroidTest**/*.kt|*.java (case-insensitive on AndroidTest; supports flavors)
        * Flutter integration tests: integration_test/**/*.dart, test_driver/**/*.dart
- Flat filenames (collision-safe) using FULL relpath:
    {owner}.{project}__{ci_platform}++{file_lower_relpath}
- Strictly NO content scanning except Manifest <activity> check.
- CSV index: owner, repo, repo_url, default_branch, commit_sha, relative_path, filename,
             flat_filename, ci_platform, html_url, saved_to, bucket, components (blank)
- Optional: Project metadata via GitHub API + paginated counts (kept as-is)
"""

import os, re, csv, random, stat, shutil, subprocess, requests, hashlib
from pathlib import Path
from urllib.parse import urlparse
import pandas as pd
from dotenv import load_dotenv, set_key

# ========= CONFIG =========
MAX_PROJECTS = 4697
RANDOM_SEED = 42
NUM_SAMPLES_TO_KEEP = 150
ENV_FILE = 'All_tokens.env'

# ---- Inputs / Outputs ----
csv_path = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_8\URL_List.csv")
base_dir = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_8")

clone_dir = base_dir / "Cloned repos"
cloned_sample_dir = base_dir / "Cloned_Sample"
config_bucket = base_dir / "All_Config_Files"
tests_bucket = base_dir / "All_Test_Files"
commits_dir = base_dir / "Commits"
git_metadata_dir = base_dir / "Git_Metadata"

# Global CSV index
flat_index_csv = config_bucket.parent / "All_Config_Index.csv"

metadata_path = base_dir / "Project_Metadata.csv"
list_of_config_path = base_dir / "List_of_Config.csv"

# ========= ENV / TOKENS =========
load_dotenv(ENV_FILE)
TOKENS = [os.getenv(f'GITHUB_TOKEN_{i}') for i in range(1, 7)]
TOKENS = [t for t in TOKENS if t]
if not TOKENS:
    print("⚠️ Warning: No GitHub tokens found in All_tokens.env (API metadata might be rate limited).")
token_index = 0

START_NUMBER = int(os.getenv("START_NUMBER") or "1")
SAMPLE_LIST_RAW = os.getenv("SAMPLE_LIST", "").strip()

# ========= Ensure folders =========
for path in [clone_dir, commits_dir, cloned_sample_dir, git_metadata_dir, config_bucket, tests_bucket]:
    path.mkdir(parents=True, exist_ok=True)

# ========= STRICT CI YAML (provider locations) =========
# (unchanged from your code)
ci_patterns = {
    r'\.travis\.ya?ml$': 'Travis_CI',
    r'\.appveyor\.ya?ml$': 'AppVeyor',
    r'appveyor\.ya?ml$': 'AppVeyor',
    r'circle\.ya?ml$': 'Circle_CI',
    r'\.circleci/config\.(yml|yaml)$': 'Circle_CI',
    r'azure-pipelines\.ya?ml$': 'Azure_Pipelines',
    r'\.github/workflows/.*\.(yml|yaml)$': 'GitHub_Actions',
    r'bitbucket-pipelines\.ya?ml$': 'Bitbucket',
    r'\.gitlab-ci\.ya?ml$': 'GitLab',
    r'Jenkinsfile\.ya?ml$': 'Jenkins',
    r'bitrise\.ya?ml$': 'Bitrise',
    r'bamboo\.ya?ml$': 'Bamboo',
    r'codeship-services\.ya?ml$': 'Codeship',
    r'\.gocd\.ya?ml$': 'GoCD',
    r'\.cirrus\.ya?ml$': 'Cirrus',
    r'wercker\.ya?ml$': 'Wercker',
    r'semaphore\.ya?ml$': 'Semaphore',
    r'codemagic\.ya?ml$': 'Nevercode',
}

# Normalize provider name -> token used in flat filename
ci_provider_token = {
    'GitHub_Actions': 'github_actions',
    'GitLab': 'gitlab',
    'Circle_CI': 'circle_ci',
    'Azure_Pipelines': 'azure_pipelines',
    'Travis_CI': 'travis_ci',
    'Bitrise': 'bitrise',
    'Bitbucket': 'bitbucket',
    'Jenkins': 'jenkins',
    'Bamboo': 'bamboo',
    'Codeship': 'codeship',
    'GoCD': 'gocd',
    'Cirrus': 'cirrus',
    'Wercker': 'wercker',
    'Semaphore': 'semaphore',
    'Nevercode': 'codemagic',
    'AppVeyor': 'appveyor',
}

# ========= Allowlisted CI JSON filenames =========
LIKELY_CI_JSON = {
    "android-studio-loading.json",
    "saucectl.config.json",
    "firebase.json",
    "test-lab.json",
}

# ========= CSV setup =========
if not flat_index_csv.exists():
    with open(flat_index_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "owner","repo","repo_url","default_branch","commit_sha",
            "relative_path","filename","flat_filename","ci_platform","html_url",
            "saved_to","bucket","components"
        ])
        writer.writeheader()

# ========= Helpers =========
def run(cmd, cwd=None, check=True):
    return subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, check=check)

def strict_yaml_match(rel_path: str):
    """Return (is_strict: bool, provider_name: str) for YAML files only."""
    p = rel_path.replace("\\", "/")
    for pattern, provider in ci_patterns.items():
        if re.search(pattern, p, re.IGNORECASE):
            return True, provider
    return False, ""

def parse_owner_repo(url: str):
    parts = urlparse(url)
    if parts.netloc.lower() != "github.com":
        raise ValueError("Only github.com URLs supported")
    pieces = parts.path.strip("/").split("/")
    if len(pieces) < 2:
        raise ValueError("Invalid GitHub URL")
    return pieces[0], pieces[1].replace(".git", "")

def detect_default_branch_via_git(repo_url: str) -> str:
    p = run(["git", "ls-remote", "--symref", repo_url, "HEAD"])
    for line in p.stdout.splitlines():
        s = line.strip()
        if s.startswith("ref: ") and s.endswith("HEAD"):
            ref = s.split()[1]
            if ref.startswith("refs/heads/"):
                return ref.split("/", 2)[2]
    for guess in ("main", "master"):
        try:
            run(["git", "ls-remote", repo_url, f"refs/heads/{guess}"], check=True)
            return guess
        except Exception:
            pass
    raise RuntimeError("Could not determine default branch (ls-remote)")

def shallow_clone_branch(repo_url: str, dest: Path, branch: str):
    dest.parent.mkdir(parents=True, exist_ok=True)
    run(['git', 'clone', '--depth', '1', '--single-branch', '--branch', branch, repo_url, str(dest)])

# ---- Classification helpers (PATH/NAME ONLY, no content except Manifest check) ----

SHELL_EXTS = {'.sh', '.bash', '.zsh', '.ksh', '.bat', '.cmd', '.ps1', '.psm1', '.psd1'}
MAKEFILES = {'makefile', 'gnumakefile', 'makefile.win', 'makefile.mak'}

def is_shell_or_make(file_path: Path) -> (bool, str):
    name = file_path.name.lower()
    ext = file_path.suffix.lower()
    if name in MAKEFILES:
        return True, 'shell'   # keep token simple
    if ext in SHELL_EXTS:
        if ext in {'.ps1', '.psm1', '.psd1'}:
            return True, 'shell_ps'
        if ext in {'.bat', '.cmd'}:
            return True, 'shell_win'
        return True, 'shell'
    return False, ''

def is_gradle_or_settings(file_path: Path) -> (bool, str):
    n = file_path.name.lower()
    if n.endswith('.gradle') or n.endswith('.gradle.kts'):
        return True, 'gradle'
    if n in {'gradle.properties', 'settings.gradle', 'settings.gradle.kts'}:
        return True, 'gradle'
    return False, ''

def is_manifest(file_path: Path) -> bool:
    return file_path.name.lower() == 'androidmanifest.xml'

def manifest_has_activity(file_path: Path) -> bool:
    try:
        txt = file_path.read_text(encoding='utf-8', errors='ignore')
        return re.search(r'<\s*activity\b', txt, re.IGNORECASE) is not None
    except Exception:
        return False

def is_allowlisted_ci_json(file_path: Path) -> (bool, str):
    n = file_path.name.lower()
    if n in LIKELY_CI_JSON:
        if n.startswith('saucectl'):
            return True, 'sauce_labs'
        if n.startswith('android-studio'):
            return True, 'android_studio'
        if n in {'firebase.json', 'test-lab.json'}:
            return True, 'firebase_test_lab'
        return True, 'ci_json'
    return False, ''

def is_flutter_pubspec(file_path: Path) -> bool:
    return file_path.name.lower() == 'pubspec.yaml'

def is_flutter_test(file_path: Path) -> (bool, str):
    rel = file_path.as_posix().lower()
    if rel.startswith('integration_test/') or '/integration_test/' in rel:
        return True, 'androidtest_dart'
    if rel.startswith('test_driver/') or '/test_driver/' in rel:
        return True, 'androidtest_dart'
    return False, ''

def is_androidtest_code(file_path: Path) -> (bool, str):
    rel = file_path.as_posix().lower()
    # must be under a src/... path AND have a segment that contains "androidtest"
    if '/src/' in rel and 'androidtest' in rel:
        if file_path.suffix.lower() == '.kt':
            return True, 'androidtest_kotlin'
        if file_path.suffix.lower() == '.java':
            return True, 'androidtest_java'
    return False, ''

def sanitize_token(s: str) -> str:
    s = s.lower()
    return re.sub(r'[^a-z0-9._+-]', '_', s)

def relpath_token(rel: str) -> str:
    rel = rel.replace('\\', '/').lower()
    rel = rel.replace('/', '__')
    rel = re.sub(r'[^a-z0-9._+\-__]', '_', rel)
    rel = re.sub(r'__+', '__', rel).strip('_')
    return rel

def short_hash(s: str) -> str:
    return hashlib.sha1(s.encode('utf-8')).hexdigest()[:8]

def make_flat_filename(owner: str, project: str, ci_platform: str, rel_path: str, used_names_set: set) -> str:
    owner_tok = sanitize_token(owner)
    project_tok = sanitize_token(project)
    ci_tok = sanitize_token(ci_platform or 'other')
    file_tok = relpath_token(rel_path)

    base = f"{owner_tok}.{project_tok}__{ci_tok}++{file_tok}"
    # Length guard
    MAXLEN = 200
    name = base if len(base) <= MAXLEN else f"{base[:MAXLEN-11]}__d{short_hash(rel_path)}"
    # Collision guard in-bucket
    if name in used_names_set:
        name = f"{name}__d{short_hash(rel_path)}"
    used_names_set.add(name)
    return name

def save_file(bucket_dir: Path, flat_filename: str, src: Path) -> Path:
    bucket_dir.mkdir(parents=True, exist_ok=True)
    dest = bucket_dir / flat_filename
    shutil.copy2(src, dest)
    return dest

# ========= Load URL list =========
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip().str.lower()
df = df[df['github_url'].notna()]
df['github_url'] = df['github_url'].astype(str).str.strip()
df = df[df['github_url'].str.startswith("https://")]
df[['github_url']].to_csv(base_dir / 'Sorted_URL_List.csv', index_label='Index')

# ========= Sampling =========
if SAMPLE_LIST_RAW:
    sample_indices_to_keep = set(map(int, SAMPLE_LIST_RAW.split(',')))
    print(f"🔁 Loaded SAMPLE_LIST from .env with {len(sample_indices_to_keep)} indices.")
else:
    random.seed(RANDOM_SEED)
    sample_indices_to_keep = set(random.sample(range(len(df)), min(NUM_SAMPLES_TO_KEEP, len(df))))
    sample_string = ",".join(map(str, sorted(sample_indices_to_keep)))
    set_key(ENV_FILE, 'SAMPLE_LIST', sample_string)
    print(f"🎲 Generated and saved new SAMPLE_LIST with {len(sample_indices_to_keep)} indices.")

# ========= Process =========
review_status_rows = []
CLONE_FAILURE_COLUMNS = ["repo_index", "repo_name", "github_url", "error_message"]

for i in range(START_NUMBER - 1, min(len(df), MAX_PROJECTS)):
    url = df.iloc[i]['github_url']
    owner_repo = urlparse(url).path.strip("/").split("/")
    if len(owner_repo) < 2:
        continue
    owner, project = owner_repo[0], owner_repo[1].replace(".git", "")
    repo_index = str(i).zfill(4)
    repo_name_tag = f"{repo_index}.{owner}.{project}"
    repo_path = clone_dir / repo_name_tag
    print(f"\n🔍 [{i+1}/{len(df)}] Processing {repo_name_tag}...")

    # Clone default branch only
    try:
        default_branch = detect_default_branch_via_git(url)
        print(f"📌 Default branch: {default_branch}")
        shallow_clone_branch(url, repo_path, default_branch)
        print("✅ Clone complete")
    except Exception as e:
        error_message = (str(e) or "Unknown error").strip()
        print(f"❌ Clone failed for {repo_name_tag}\n{error_message}")
        review_status_rows.append({"html_url": url.strip(), "clone_status": "no", "yml_detected": "no"})
        pd.DataFrame([review_status_rows[-1]]).to_csv(
            base_dir / "Clone_Status.csv", mode='a', header=not (base_dir / "Clone_Status.csv").exists(), index=False
        )
        fail_row = {"repo_index": repo_index, "repo_name": repo_name_tag, "github_url": url.strip(), "error_message": error_message}
        fail_path = base_dir / "Clone_Failures.csv"
        pd.DataFrame([fail_row])[CLONE_FAILURE_COLUMNS].to_csv(
            fail_path, mode='a', header=not fail_path.exists(), index=False
        )
        continue

    # Commit count + SHA
    try:
        local_commit_count = int(subprocess.run(['git', '-C', str(repo_path), 'rev-list', '--count', 'HEAD'],
                                               capture_output=True, text=True, check=True).stdout.strip())
    except subprocess.CalledProcessError:
        local_commit_count = 0
        print(f"⚠️ Could not get commit count for {repo_name_tag}")

    try:
        head_sha = subprocess.run(['git', '-C', str(repo_path), 'rev-parse', '--verify', 'HEAD'],
                                  capture_output=True, text=True, check=True).stdout.strip()
    except subprocess.CalledProcessError:
        head_sha = ""

    # Optional per-repo commit metadata CSV (kept)
    if local_commit_count > 0:
        try:
            cmd_hashes = ["git", "-C", str(repo_path), "log", "--pretty=format:%H"]
            result_hashes = subprocess.run(cmd_hashes, capture_output=True, text=True, check=True)
            commit_hashes = result_hashes.stdout.strip().split("\n")
            rows = []
            for commit in commit_hashes:
                cmd_metadata = ["git", "-C", str(repo_path), "show", "--quiet",
                                f"--pretty=format:%H|%an|%ae|%ad|%s", "--date=iso", commit]
                result_metadata = subprocess.run(cmd_metadata, capture_output=True, text=True)
                if not result_metadata.stdout:
                    continue
                parts = result_metadata.stdout.strip().split("|", maxsplit=4)
                if len(parts) < 5:
                    continue
                rows.append({
                    "commit_hash": parts[0],
                    "author_name": parts[1],
                    "author_email": parts[2],
                    "commit_date": parts[3],
                    "commit_message": parts[4],
                })
            if rows:
                dfc = pd.DataFrame(rows)
                git_metadata_dir.mkdir(parents=True, exist_ok=True)
                flat_filename = f"{project}__GitMetadata++contributors_commits.csv"
                dfc.to_csv(git_metadata_dir / flat_filename, index=False)
        except subprocess.CalledProcessError as e:
            print(f"❌ Failed to extract commit data for {repo_path.name}: {e}")

    # Walk files and extract
    any_yml = False
    legacy_config_rows = []

    # Collision + symlink guards per bucket
    used_names_config = set()
    used_names_tests = set()
    seen_realpaths = set()

    for root, _, files in os.walk(repo_path):
        for file in files:
            file_path = Path(root) / file
            try:
                realp = file_path.resolve()
            except Exception:
                realp = file_path
            if realp in seen_realpaths:
                continue
            seen_realpaths.add(realp)

            rel_path = str(file_path.relative_to(repo_path)).replace("\\", "/")
            rel_lower = rel_path.lower()
            filename_lower = file.lower()

            # -------- 1) STRICT CI YAML (CONFIG) --------
            if filename_lower.endswith(('.yml', '.yaml')):
                is_strict, provider = strict_yaml_match(rel_path)
                if not is_strict:
                    continue
                ci_platform = ci_provider_token.get(provider, 'ci_yaml')
                flat_filename = make_flat_filename(owner, project, ci_platform, rel_path, used_names_config)
                dest_path = save_file(config_bucket, flat_filename, file_path)
                any_yml = True
                bucket = "All_Config_Files"

            # -------- 2) SHELL / MAKEFILES (CONFIG) --------
            elif is_shell_or_make(file_path)[0]:
                _, ci_platform = is_shell_or_make(file_path)
                flat_filename = make_flat_filename(owner, project, ci_platform, rel_path, used_names_config)
                dest_path = save_file(config_bucket, flat_filename, file_path)
                bucket = "All_Config_Files"

            # -------- 3) GRADLE & SETTINGS (CONFIG) --------
            elif is_gradle_or_settings(file_path)[0]:
                _, ci_platform = is_gradle_or_settings(file_path)
                flat_filename = make_flat_filename(owner, project, ci_platform, rel_path, used_names_config)
                dest_path = save_file(config_bucket, flat_filename, file_path)
                bucket = "All_Config_Files"

            # -------- 4) ANDROID MANIFEST with <activity> (CONFIG) --------
            elif is_manifest(file_path):
                if manifest_has_activity(file_path):
                    ci_platform = 'manifest_activity'
                    flat_filename = make_flat_filename(owner, project, ci_platform, rel_path, used_names_config)
                    dest_path = save_file(config_bucket, flat_filename, file_path)
                    bucket = "All_Config_Files"
                else:
                    continue  # skip manifests without an activity

            # -------- 5) ALLOWLISTED CI JSON (CONFIG) --------
            elif is_allowlisted_ci_json(file_path)[0]:
                _, ci_platform = is_allowlisted_ci_json(file_path)
                flat_filename = make_flat_filename(owner, project, ci_platform, rel_path, used_names_config)
                dest_path = save_file(config_bucket, flat_filename, file_path)
                bucket = "All_Config_Files"

            # -------- 6) FLUTTER pubspec.yaml (CONFIG) --------
            elif is_flutter_pubspec(file_path):
                ci_platform = 'flutter'
                flat_filename = make_flat_filename(owner, project, ci_platform, rel_path, used_names_config)
                dest_path = save_file(config_bucket, flat_filename, file_path)
                bucket = "All_Config_Files"

            # -------- 7) FLUTTER integration tests (TESTS) --------
            elif is_flutter_test(file_path)[0]:
                _, ci_platform = is_flutter_test(file_path)
                flat_filename = make_flat_filename(owner, project, ci_platform, rel_path, used_names_tests)
                dest_path = save_file(tests_bucket, flat_filename, file_path)
                bucket = "All_Test_Files"

            # -------- 8) ANDROIDTEST code (.kt/.java) (TESTS) --------
            elif is_androidtest_code(file_path)[0]:
                _, ci_platform = is_androidtest_code(file_path)
                flat_filename = make_flat_filename(owner, project, ci_platform, rel_path, used_names_tests)
                dest_path = save_file(tests_bucket, flat_filename, file_path)
                bucket = "All_Test_Files"

            else:
                continue  # nothing to save for this file

            # Write unified CSV index
            with open(flat_index_csv, "a", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=[
                    "owner","repo","repo_url","default_branch","commit_sha",
                    "relative_path","filename","flat_filename","ci_platform","html_url",
                    "saved_to","bucket","components"
                ])
                writer.writerow({
                    "owner": owner,
                    "repo": project,
                    "repo_url": url.strip(),
                    "default_branch": default_branch,
                    "commit_sha": head_sha,
                    "relative_path": rel_path,
                    "filename": file,
                    "flat_filename": flat_filename,
                    "ci_platform": ci_platform,
                    "html_url": f"https://github.com/{owner}/{project}/blob/{default_branch}/{rel_path}",
                    "saved_to": str(dest_path),
                    "bucket": bucket,
                    "components": ""  # content analysis deferred
                })

            # Maintain legacy List_of_Config.csv (kept)
            legacy_config_rows.append({
                "html_url": url.strip().rstrip('/'),
                "repo_name": repo_name_tag,
                "config_file_path": flat_filename,
                "original_rel_path": rel_path,
                "file_name": file,
                "file_type": filename_lower.split('.')[-1] if '.' in filename_lower else filename_lower
            })

    # Clone status
    review_status_row = {"html_url": url.strip(), "clone_status": "yes", "yml_detected": "yes" if any_yml else "no"}
    pd.DataFrame([review_status_row]).to_csv(
        base_dir / "Clone_Status.csv", mode='a', header=not (base_dir / "Clone_Status.csv").exists(), index=False
    )

    # Persist legacy List_of_Config.csv
    if legacy_config_rows:
        ldf = pd.DataFrame(legacy_config_rows)
        if list_of_config_path.exists():
            ldf.to_csv(list_of_config_path, mode='a', header=False, index=False)
        else:
            ldf.to_csv(list_of_config_path, mode='w', header=True, index=False)

    # Project metadata + paginated counts + contributors (kept)
    try:
        headers = {}
        if TOKENS:
            headers = {'Authorization': f'token {TOKENS[token_index % len(TOKENS)]}'}
            token_index += 1
        base_api = f"https://api.github.com/repos/{owner}/{project}"
        r = requests.get(base_api, headers=headers, timeout=30)
        data = r.json() if r.status_code == 200 else {}

        if TOKENS:
            headers = {'Authorization': f'token {TOKENS[token_index % len(TOKENS)]}'}
            token_index += 1
        def get_count(api_url, headers):
            per_page = 100
            page = 1
            total_items = 0
            try:
                while True:
                    response = requests.get(api_url, headers=headers, params={"per_page": per_page, "page": page}, timeout=30)
                    if response.status_code != 200:
                        break
                    items = response.json()
                    if not isinstance(items, list):
                        break
                    total_items += len(items)
                    if len(items) < per_page:
                        break
                    page += 1
            except Exception:
                pass
            return total_items

        contributors_count = get_count(f"{base_api}/contributors", headers)
        if TOKENS:
            headers = {'Authorization': f'token {TOKENS[token_index % len(TOKENS)]}'}
            token_index += 1
        pulls_count = get_count(f"{base_api}/pulls?state=all", headers)
        if TOKENS:
            headers = {'Authorization': f'token {TOKENS[token_index % len(TOKENS)]}'}
            token_index += 1
        commits_count = get_count(f"{base_api}/commits", headers)

        metadata_row = {
            "html_url": url,
            "repo_index": repo_index,
            "repo_name": repo_name_tag,
            "id": data.get("id"),
            "name": data.get("name"),
            "full_name": data.get("full_name"),
            "owner": data.get("owner", {}).get("login") if data.get("owner") else None,
            "private": data.get("private"),
            "fork": data.get("fork"),
            "created_at": data.get("created_at"),
            "updated_at": data.get("updated_at"),
            "pushed_at": data.get("pushed_at"),
            "homepage": data.get("homepage"),
            "size": data.get("size"),
            "stargazers_count": data.get("stargazers_count"),
            "language": data.get("language"),
            "forks_count": data.get("forks_count"),
            "open_issues_count": data.get("open_issues_count"),
            "license": data.get("license", {}).get("name") if data.get("license") else None,
            "topics": ", ".join(data.get("topics", [])) if data.get("topics") else None,
            "visibility": data.get("visibility"),
            "default_branch": data.get("default_branch"),
            "has_issues": data.get("has_issues"),
            "has_projects": data.get("has_projects"),
            "has_downloads": data.get("has_downloads"),
            "has_wiki": data.get("has_wiki"),
            "has_pages": data.get("has_pages"),
            "archived": data.get("archived"),
            "disabled": data.get("disabled"),
            "allow_forking": data.get("allow_forking"),
            "is_template": data.get("is_template"),
            "web_commit_signoff_required": data.get("web_commit_signoff_required"),
            "contributors": contributors_count,
            "pull_requests": pulls_count,
            "commits_GitAPI": commits_count,
            "local_commit_count": local_commit_count
        }
        mdf = pd.DataFrame([metadata_row])
        if metadata_path.exists():
            mdf.to_csv(metadata_path, mode='a', header=False, index=False)
        else:
            mdf.to_csv(metadata_path, mode='w', header=True, index=False)

        if TOKENS:
            headers = {'Authorization': f'token {TOKENS[token_index % len(TOKENS)]}'}
            token_index += 1
        contrib_url = f"{base_api}/contributors"
        r_contrib = requests.get(contrib_url, headers=headers, timeout=30)
        if r_contrib.status_code == 200:
            contributor_logins = [c['login'] for c in r_contrib.json()]
            contributors_text = "\n".join(contributor_logins)
            contributors_filename = f"{owner}.{project}__Contributors++list.txt"
            contributors_path = commits_dir / contributors_filename
            with open(contributors_path, "w", encoding="utf-8") as f:
                f.write(contributors_text)

    except Exception as e:
        print(f"⚠️ Metadata or contributors error for {repo_name_tag}: {e}")

    # Cleanup or keep sample
    try:
        if i in sample_indices_to_keep:
            dest_path = cloned_sample_dir / repo_path.name
            if dest_path.exists():
                shutil.rmtree(dest_path, ignore_errors=True)
            shutil.move(str(repo_path), str(dest_path))
            print(f"📆 Sample repo moved to: {dest_path}")
        else:
            shutil.rmtree(repo_path, onerror=lambda f,p,e: (os.chmod(p, stat.S_IWRITE), f(p)))
            print(f"🕵️ Deleted cloned repo: {repo_name_tag}")
    except Exception as e:
        print(f"❌ Error handling repo folder for {repo_name_tag}: {e}")

    set_key(ENV_FILE, 'START_NUMBER', str(i + 2))

# Final dedupes
for p in [base_dir / "List_of_Config.csv",
          base_dir / "Clone_Failures.csv",
          base_dir / "Project_Metadata.csv",
          base_dir / "Clone_Status.csv"]:
    if p.exists():
        try:
            dfp = pd.read_csv(p)
            dfp.drop_duplicates().to_csv(p, index=False)
        except Exception:
            pass

print("\n✅ Process complete.")


🔁 Loaded SAMPLE_LIST from .env with 150 indices.

🔍 [1/4697] Processing 0000.jamplus.jamplus...
📌 Default branch: master
✅ Clone complete
🕵️ Deleted cloned repo: 0000.jamplus.jamplus

🔍 [2/4697] Processing 0001.samuelclay.NewsBlur...
📌 Default branch: master
✅ Clone complete
🕵️ Deleted cloned repo: 0001.samuelclay.NewsBlur

🔍 [3/4697] Processing 0002.connectbot.connectbot...
📌 Default branch: main
✅ Clone complete
🕵️ Deleted cloned repo: 0002.connectbot.connectbot

🔍 [4/4697] Processing 0003.pocmo.Yaaic...
📌 Default branch: master
✅ Clone complete
🕵️ Deleted cloned repo: 0003.pocmo.Yaaic

🔍 [5/4697] Processing 0004.Ramblurr.Anki-Android...
📌 Default branch: feature-multimedia-editor
✅ Clone complete
📆 Sample repo moved to: C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_8\Cloned_Sample\0004.Ramblurr.Anki-Android

🔍 [6/4697] Processing 0005.XCSoar.XCSoar...
📌 Default branch: master
✅ Clone complete
🕵️ Deleted cloned repo: 0005.XCSoar.XCSoar

🔍 [7/4697] Processing 0006.gradle.gradle...

Exception in thread Thread-149 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\threading.py", line 1009, in _bootstrap_inner
    self.run()
  File "c:\GitHub\.PCvenv\lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\threading.py", line 946, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 1494, in _readerthread
    buffer.append(fh.read())
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
UnicodeDecodeError: 'charmap' codec can't decode byte 0x8d in position 56: character maps to <undefined>


✅ Clone complete
🕵️ Deleted cloned repo: 0015.RHVoice.RHVoice

🔍 [17/4697] Processing 0016.NXT.LEGO-MINDSTORMS-MINDdroid...
📌 Default branch: master
✅ Clone complete
🕵️ Deleted cloned repo: 0016.NXT.LEGO-MINDSTORMS-MINDdroid

🔍 [18/4697] Processing 0017.opendocument-app.OpenDocument.droid...
📌 Default branch: main
✅ Clone complete
🕵️ Deleted cloned repo: 0017.opendocument-app.OpenDocument.droid

🔍 [19/4697] Processing 0018.drawpile.Drawpile...
📌 Default branch: main
✅ Clone complete
🕵️ Deleted cloned repo: 0018.drawpile.Drawpile

🔍 [20/4697] Processing 0019.guardianproject.ObscuraCam...
📌 Default branch: master
✅ Clone complete
🕵️ Deleted cloned repo: 0019.guardianproject.ObscuraCam

🔍 [21/4697] Processing 0020.maxpower47.PinDroid...
📌 Default branch: master
✅ Clone complete
🕵️ Deleted cloned repo: 0020.maxpower47.PinDroid

🔍 [22/4697] Processing 0021.chesterbr.minitruco-android...
📌 Default branch: main
✅ Clone complete
🕵️ Deleted cloned repo: 0021.chesterbr.minitruco-android

🔍 [23/4

Exception in thread Thread-2077 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\threading.py", line 1009, in _bootstrap_inner
    self.run()
  File "c:\GitHub\.PCvenv\lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\threading.py", line 946, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 1494, in _readerthread
    buffer.append(fh.read())
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
UnicodeDecodeError: 'charmap' codec can't decode byte 0x8d in position 100: character maps to <undefined>


✅ Clone complete
🕵️ Deleted cloned repo: 0214.daimajia.AnimeTaste

🔍 [216/4697] Processing 0215.novoda.spikes...
📌 Default branch: master
✅ Clone complete
🕵️ Deleted cloned repo: 0215.novoda.spikes

🔍 [217/4697] Processing 0216.stephanenicolas.boundbox...
📌 Default branch: master
✅ Clone complete
🕵️ Deleted cloned repo: 0216.stephanenicolas.boundbox

🔍 [218/4697] Processing 0217.FeatureIDE.FeatureIDE...
📌 Default branch: develop
❌ Clone failed for 0217.FeatureIDE.FeatureIDE
Command '['git', 'clone', '--depth', '1', '--single-branch', '--branch', 'develop', 'https://github.com/FeatureIDE/FeatureIDE', 'C:\\Android Mobile App\\Step2_Clone_Repo\\Type_1\\Aug_8\\Cloned repos\\0217.FeatureIDE.FeatureIDE']' returned non-zero exit status 128.

🔍 [219/4697] Processing 0218.452.USBHIDTerminal...
📌 Default branch: master
✅ Clone complete
🕵️ Deleted cloned repo: 0218.452.USBHIDTerminal

🔍 [220/4697] Processing 0219.simlar.simlar-android...
📌 Default branch: master
✅ Clone complete
🕵️ Deleted cloned

...

✅ Clone complete
🕵️ Deleted cloned repo: 2700.alvr.katana

🔍 [2702/4697] Processing 2701.joreilly.WordMasterKMP...
📌 Default branch: main
✅ Clone complete
🕵️ Deleted cloned repo: 2701.joreilly.WordMasterKMP

🔍 [2703/4697] Processing 2702.2BAB.Koncat...
📌 Default branch: main
✅ Clone complete
🕵️ Deleted cloned repo: 2702.2BAB.Koncat

🔍 [2704/4697] Processing 2703.rafsanjani.datepickertimeline...
📌 Default branch: main
✅ Clone complete
🕵️ Deleted cloned repo: 2703.rafsanjani.datepickertimeline

🔍 [2705/4697] Processing 2704.Ashinch.ReadYou...
📌 Default branch: main
✅ Clone complete
🕵️ Deleted cloned repo: 2704.Ashinch.ReadYou

🔍 [2706/4697] Processing 2705.HighCapable.YukiHookAPI...
📌 Default branch: master
✅ Clone complete
🕵️ Deleted cloned repo: 2705.HighCapable.YukiHookAPI

🔍 [2707/4697] Processing 2706.RyensX.MediaBox...
📌 Default branch: dev
✅ Clone complete
🕵️ Deleted cloned repo: 2706.RyensX.MediaBox

🔍 [2708/4697] Processing 2707.saket.swipe...
📌 Default branch: trunk
✅ Clone com

Exception in thread Thread-26109 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\threading.py", line 1009, in _bootstrap_inner
    self.run()
  File "c:\GitHub\.PCvenv\lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\threading.py", line 946, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 1494, in _readerthread
    buffer.append(fh.read())
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
UnicodeDecodeError: 'charmap' codec can't decode byte 0x81 in position 134: character maps to <undefined>


✅ Clone complete
🕵️ Deleted cloned repo: 2724.GuoguoDad.jd_mall

🔍 [2726/4697] Processing 2725.rodit.SnapMod...
📌 Default branch: master
✅ Clone complete
🕵️ Deleted cloned repo: 2725.rodit.SnapMod

🔍 [2727/4697] Processing 2726.fankes.ColorOSNotifyIcon...
📌 Default branch: master
✅ Clone complete
🕵️ Deleted cloned repo: 2726.fankes.ColorOSNotifyIcon

🔍 [2728/4697] Processing 2727.google-developer-training.basic-android-kotlin-compose-birthday-card-app...
📌 Default branch: main
✅ Clone complete
🕵️ Deleted cloned repo: 2727.google-developer-training.basic-android-kotlin-compose-birthday-card-app

🔍 [2729/4697] Processing 2728.Infomaniak.android-kMail...
📌 Default branch: main
✅ Clone complete
🕵️ Deleted cloned repo: 2728.Infomaniak.android-kMail

🔍 [2730/4697] Processing 2729.x13a.Sentry...
📌 Default branch: main
✅ Clone complete
🕵️ Deleted cloned repo: 2729.x13a.Sentry

🔍 [2731/4697] Processing 2730.Zomato.sushi-ui-android...
📌 Default branch: dev
✅ Clone complete
🕵️ Deleted cloned repo

Exception in thread Thread-26977 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\threading.py", line 1009, in _bootstrap_inner
    self.run()
  File "c:\GitHub\.PCvenv\lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\threading.py", line 946, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 1494, in _readerthread
    buffer.append(fh.read())
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
UnicodeDecodeError: 'charmap' codec can't decode byte 0x8d in position 66: character maps to <undefined>


✅ Clone complete
🕵️ Deleted cloned repo: 2814.hoc081098.GithubSearchKMM-Compose-SwiftUI

🔍 [2816/4697] Processing 2815.SmartToolFactory.Compose-BeforeAfter...
📌 Default branch: master
✅ Clone complete
🕵️ Deleted cloned repo: 2815.SmartToolFactory.Compose-BeforeAfter

🔍 [2817/4697] Processing 2816.MateusRodCosta.SaveLocally...
📌 Default branch: main
✅ Clone complete
🕵️ Deleted cloned repo: 2816.MateusRodCosta.SaveLocally

🔍 [2818/4697] Processing 2817.ANSSI-FR.ultrablue...
📌 Default branch: dev
✅ Clone complete
📆 Sample repo moved to: C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_8\Cloned_Sample\2817.ANSSI-FR.ultrablue

🔍 [2819/4697] Processing 2818.KunMinX.MVI-Dispatcher-KTX...
📌 Default branch: main
✅ Clone complete
🕵️ Deleted cloned repo: 2818.KunMinX.MVI-Dispatcher-KTX

🔍 [2820/4697] Processing 2819.Wavesonics.hammer-editor...
📌 Default branch: develop
✅ Clone complete
🕵️ Deleted cloned repo: 2819.Wavesonics.hammer-editor

🔍 [2821/4697] Processing 2820.xihan123.AGE...
📌 Default 